**Mount Drive**

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Import Libraries**

In [17]:
import os
import joblib
import pandas as pd
import numpy as np

**Load Transformer Model + Scaler + Data**

In [19]:
import joblib

SCALER_PATH = "/content/drive/MyDrive/c01-price-forecasting/data/model_inputs/lstm/scaler.pkl"
scaler = joblib.load(SCALER_PATH)

print("Scaler loaded ✅")

Scaler loaded ✅


In [20]:
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization, Dense, Dropout, Input
from tensorflow.keras.models import Model
import tensorflow as tf

MODEL_PATH = "/content/drive/MyDrive/c01-price-forecasting/models/transformer_model.h5"

try:
    model_tr = load_model(MODEL_PATH, compile=False)
    print("✅ Model loaded (normal)")

except:
    print("⚠️ Trying fallback...")

    def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
        x = MultiHeadAttention(num_heads=num_heads, key_dim=head_size)(inputs, inputs)
        x = Dropout(dropout)(x)
        x = LayerNormalization(epsilon=1e-6)(x)

        x_ff = Dense(ff_dim, activation="relu")(x)
        x_ff = Dense(inputs.shape[-1])(x_ff)
        x = x + x_ff
        x = LayerNormalization(epsilon=1e-6)(x)

        return x

    input_shape = (12, len(scaler.feature_names_in_) - 1)

    inputs = Input(shape=input_shape)

    x = transformer_encoder(inputs, head_size=64, num_heads=4, ff_dim=128, dropout=0.1)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(0.2)(x)
    outputs = Dense(1)(x)

    model_tr = Model(inputs, outputs)

    model_tr.load_weights(MODEL_PATH)

    print("✅ Model loaded (fallback rebuild)")

⚠️ Trying fallback...
✅ Model loaded (fallback rebuild)


In [21]:
import pandas as pd

DATA_PATH = "/content/drive/MyDrive/c01-price-forecasting/data/processed/cleaned_data.csv"

df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])

print("Dataset loaded ✅")
print(df.head())

Dataset loaded ✅
        date district        paddy_type  min_price  max_price  avg_price  \
0 2015-03-26   Ampara  Long_Grain_White       34.0       40.0      37.14   
1 2015-04-02   Ampara  Long_Grain_White       34.0       40.0      36.40   
2 2015-04-09   Ampara  Long_Grain_White       34.0       39.0      35.68   
3 2015-04-16   Ampara  Long_Grain_White       31.0       37.0      34.97   
4 2015-04-23   Ampara  Long_Grain_White       30.0       38.0      34.03   

   production_total  price_range  week_of_year  week_sin  ...  year  month  \
0            307661          6.0            13  1.000000  ...  2015      3   
1            309335          6.0            14  0.992709  ...  2015      4   
2            309335          5.0            15  0.970942  ...  2015      4   
3            309335          6.0            16  0.935016  ...  2015      4   
4            309335          8.0            17  0.885456  ...  2015      4   

   week  lag_1  lag_2  lag_4  lag_12  rolling_mean_4  rol

**Prepare Input**

In [22]:
def prepare_transformer_input(df_d, scaler, feature_cols, timesteps=12):

    df_last = df_d.tail(timesteps)

    X = df_last[feature_cols].values

    full = np.zeros((len(X), len(feature_cols) + 1))
    full[:, :-1] = X

    scaled = scaler.transform(full)[:, :-1]

    # Transformer expects same shape (batch, time, features)
    X_tr = scaled.reshape(1, timesteps, len(feature_cols))

    return X_tr

**Prediction Function**

In [23]:
def predict_price_transformer(district, date):

    date = pd.to_datetime(date)

    df_d = df[df['district'] == district].copy()
    df_d = df_d.sort_values('date')
    df_d = df_d[df_d['date'] < date]

    feature_cols = list(scaler.feature_names_in_[:-1])

    # Prepare input
    X_tr = prepare_transformer_input(df_d, scaler, feature_cols)

    # Predict
    y_pred_scaled = model_tr.predict(X_tr)[0][0]

    # Inverse scaling
    dummy = np.zeros((1, len(feature_cols) + 1))
    dummy[0, -1] = y_pred_scaled

    y_pred = scaler.inverse_transform(dummy)[0, -1]

    print(f"\n📍 District: {district}")
    print(f"📅 Date: {date}")
    print(f"💰 Transformer Predicted Price: {y_pred:.2f}")

    return float(y_pred)

**Test Section**

In [24]:
predict_price_transformer("Anuradhapura", "2026-05-01")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step

📍 District: Anuradhapura
📅 Date: 2026-05-01 00:00:00
💰 Transformer Predicted Price: 106.50


106.50244104266166

**Forecast Function**

In [25]:
from datetime import timedelta

def forecast_transformer(district, start_date, weeks=4):

    start_date = pd.to_datetime(start_date)

    df_d = df[df['district'] == district].copy()
    df_d = df_d.sort_values('date')
    df_d = df_d[df_d['date'] < start_date]

    df_future = df_d.copy()
    predictions = []

    feature_cols = list(scaler.feature_names_in_[:-1])

    current_date = start_date

    for i in range(weeks):

        X_tr = prepare_transformer_input(df_future, scaler, feature_cols)

        y_pred_scaled = model_tr.predict(X_tr)[0][0]

        # Inverse scaling
        dummy = np.zeros((1, len(feature_cols) + 1))
        dummy[0, -1] = y_pred_scaled
        y_pred = scaler.inverse_transform(dummy)[0, -1]

        predictions.append({
            "date": current_date,
            "predicted_price": float(y_pred)
        })

        # Add prediction back
        new_row = df_future.iloc[-1].copy()
        new_row['date'] = current_date
        new_row['avg_price'] = y_pred

        df_future = pd.concat([df_future, pd.DataFrame([new_row])], ignore_index=True)

        current_date += timedelta(weeks=1)

    return pd.DataFrame(predictions)

**Test Section**

In [26]:
forecast_transformer("Anuradhapura", "2026-05-01", weeks=8)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step


,date,predicted_price
0,2026-05-01,106.502441
1,2026-05-08,105.271359
2,2026-05-15,103.626213
3,2026-05-22,102.362770
4,2026-05-29,101.452245
5,2026-06-05,100.647317
6,2026-06-12,100.178557
7,2026-06-19,99.697590
